# 04 · Panel Veri: Sabit Etkiler mi, Rassal Etkiler mi?

Aynı birimleri (firma, il, ülke) birden çok dönem gözlediğimizde **gözlenmeyen ama zamanla değişmeyen** farkları (yönetim kalitesi, kültür, coğrafya) kontrol edebiliriz. Bu, panel verinin en büyük gücüdür.

**Öğreneceklerin**
1. Havuzlanmış EKK, sabit etkiler ve rassal etkiler modellerini kurmak
2. F, Breusch-Pagan LM ve Hausman testleriyle model seçmek
3. Kümelenmiş standart hata ve zaman etkilerini kullanmak
4. Panel analizini tez bölümü olarak raporlamak

> ⚠️ Kullanılan veri **simüle edilmiştir**. Gerçek modelde firma etkisi sermaye ile ilişkilidir; bu yüzden rassal etkiler tutarsızdır.

In [1]:
import ekonometri as ek

veri = ek.veri.yukle("firma")
veri.head()

,firma,yil,satis,calisan,sermaye,arge,ihracatci,sektor
0,F001,2015,1545000.0,64,108000.0,1.28,Hayır,Ticaret
1,F001,2016,1487000.0,61,91000.0,2.34,Hayır,Ticaret
2,F001,2017,1253000.0,64,96000.0,0.69,Hayır,Ticaret
3,F001,2018,1195000.0,58,106000.0,2.06,Hayır,Ticaret
4,F001,2019,1305000.0,63,122000.0,3.16,Hayır,Ticaret


In [2]:
print(ek.veri.aciklama("firma"))

📁 firma — Firma üretim fonksiyonu (panel veri)
⚠️ Simüle veri: gerçek gözlem değildir, eğitim amaçlıdır.

Konu: Panel veri: havuzlanmış EKK, sabit ve rassal etkiler, F / Breusch-Pagan LM / Hausman testleri, kümelenmiş standart hata

Değişkenler:
  • firma: Firma kimliği (150 firma) — birim
  • yil: Yıl (2015–2024) — zaman
  • satis: Yıllık satış (TL)
  • calisan: Çalışan sayısı
  • sermaye: Sermaye stoku (TL)
  • arge: Ar-Ge harcamasının satışa oranı (%)
  • ihracatci: İhracatçı firma mı (Evet / Hayır; zamanla değişmez)
  • sektor: İmalat / Hizmet / Ticaret (zamanla değişmez)

Veriyi üreten gerçek model (cevap anahtarı):
  log(satis) = aᵢ + 0,62·log(calisan) + 0,31·log(sermaye) + 0,045·arge + 0,15·ihracatçı + sektör + yıl etkisi + u;  firma etkisi aᵢ sermaye ve çalışanla ilişkilidir (rassal etkiler tutarsız, Hausman testi sabit etkileri seçmeli).

Örnek:
  ek.panel_analizi("np.log(satis) ~ np.log(calisan) + np.log(sermaye) + arge", veri, birim="firma", zaman="yil")


## 1. Tek komutla panel analizi

`ek.panel_analizi()` üç modeli tahmin eder, seçim testlerini uygular ve uygun modeli önerir:

In [3]:
formul = "np.log(satis) ~ np.log(calisan) + np.log(sermaye) + arge + ihracatci"
analiz = ek.panel_analizi(formul, veri, birim="firma", zaman="yil")
analiz

📊 Panel veri analizi

════════════════════════════════════════════════════════════════════════
Değişken               Havuzlanmış EKK    Sabit etkiler   Rassal etkiler
────────────────────────────────────────────────────────────────────────
ihracatci: Evet               0,1379**                          0,1374**
                              (0,0563)                          (0,0603)
ln(calisan)                  0,7502***        0,7585***        0,7588***
                              (0,0403)         (0,0283)         (0,0240)
ln(sermaye)                  0,6905***        0,4698***        0,5014***
                              (0,0414)         (0,0196)         (0,0177)
arge                         0,0540***        0,0516***        0,0520***
                              (0,0094)         (0,0035)         (0,0035)
Sabit                        2,6374***        5,2458***        4,8250***
                              (0,4864)         (0,2111)         (0,1979)
────────────────────────────────────────────────────────────────────────
Gözlem sayısı                    1.500            1.500            1.500
Birim sayısı                       150              150              150
R² (içsel)                       0,586            0,599            0,599
R² (birimler arası)              0,822            0,796            0,807
R² (genel)                       0,807            0,784            0,794
Birim etkileri                     Yok            Sabit           Rassal
Zaman etkileri                     Yok              Yok              Yok
════════════════════════════════════════════════════════════════════════
(std. hata)   *** p < 0,01; ** p < 0,05; * p < 0,10.

🔎 Model seçim testleri
F testi (havuzlanmış EKK – sabit etkiler): istatistik = 70,697 [149, 1347], p < 0,0001
   H0: Birim etkileri yoktur (havuzlanmış EKK uygundur).
   p < 0,05: H0 reddedildi — birim etkileri anlamlı; sabit etkiler havuzlanmış EKK'ye tercih edilir.
Breusch-Pagan LM testi (havuzlanmış EKK – rassal etkiler): istatistik = 4.955,740 [1], p < 0,0001
   H0: Birim etkilerinin varyansı sıfırdır (havuzlanmış EKK uygundur).
   p < 0,05: H0 reddedildi — rassal birim etkileri var; rassal etkiler havuzlanmış EKK'ye tercih edilir.
Hausman testi (sabit etkiler – rassal etkiler): istatistik = 22,091 [3], p < 0,0001
   H0: Birim etkileri açıklayıcı değişkenlerle ilişkisizdir (rassal etkiler tutarlıdır).
   p < 0,05: H0 reddedildi — birim etkileri açıklayıcı değişkenlerle ilişkili; rassal etkiler tutarsız, sabit etkiler tercih edilir. (Kovaryans farkı pozitif tanımlı olmadığından iki kovaryans matrisi de rassal etkiler hata varyansıyla ölçeklendi — Stata 'sigmamore'.)

👉 Önerilen model: Sabit etkiler (birimlere göre kümelenmiş standart hatalarla)

**Ne gördük?**
- Havuzlanmış EKK sermayenin etkisini olduğundan çok büyük tahmin ediyor: iyi yönetilen firmalar hem daha çok yatırım yapıyor hem daha çok satıyor (atlanmış değişken yanlılığı).
- Hausman testi rassal etkileri reddediyor: firma etkisi açıklayıcı değişkenlerle ilişkili.
- `ihracatci` zamanla değişmediği için sabit etkiler modelinde tahmin edilemiyor.

## 2. Zaman etkileri: iki yönlü model

Tüm firmaları aynı anda etkileyen trendler ve şoklar (2020 gibi) ayıklanmazsa katsayılara karışır. Gerçek parametreler 0,62 (çalışan) ve 0,31 (sermaye):

In [4]:
iki_yonlu = ek.panel(formul, veri, birim="firma", zaman="yil", yontem="sabit", zaman_etkisi=True, kovaryans="kume")
iki_yonlu

══════════════════════════════════════════════════════════════════════════════
 Panel Veri — Sabit etkiler · bağımlı değişken: ln(satis)
══════════════════════════════════════════════════════════════════════════════
 Gözlem: 1500     Birim: 150    Dönem: 10    Panel: dengeli
 R² içsel: 0,6624    arası: 0,7714    genel: 0,7586    F(12, 149) = 228,36 (p <0,0001)
 Standart hatalar: birimlere göre kümelenmiş (150 küme; değişen varyans ve birim içi otokorelasyona dayanıklı) · zaman etkileri dahil
──────────────────────────────────────────────────────────────────────────────
 Değişken                         Katsayı   Std. hata        t         p
──────────────────────────────────────────────────────────────────────────────
 Sabit                             7,6239      0,2653    28,74   <0,0001 ***
 ln(calisan)                       0,6255      0,0265    23,56   <0,0001 ***
 ln(sermaye)                       0,3018      0,0226    13,33   <0,0001 ***
 arge                              0,0516

In [5]:
iki_yonlu.yorumla()

📊 Yorum — 'satis' (logaritması)

▸ Model geneli
  • Sabit etkiler modeli: 150 birim, 10 dönem, 1.500 gözlem (dengeli panel).
  • R² içsel = 0,662 (birim içi zamana bağlı değişimin açıklanan kısmı), birimler arası = 0,771, genel = 0,759.
  • Standart hatalar: birimlere göre kümelenmiş (150 küme; değişen varyans ve birim içi otokorelasyona dayanıklı).

▸ Katsayılar (diğer değişkenler sabitken)
  • 'calisan' %1 artınca 'satis' yaklaşık %0,625 artıyor (esneklik = 0,625) — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'sermaye' %1 artınca 'satis' yaklaşık %0,302 artıyor (esneklik = 0,302) — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'arge' 1 artınca 'satis' ortalama %5,29 artıyor — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).

▸ Notlar
  • Sabit etkiler katsayıları, aynı birimin zaman içindeki değişimine dayanır (birim içi etki); zamanla değişmeyen özellikler tahmin edilemez.
  • Bağımlı değişken logaritmik: katsayılar yüzde değişim olarak yorumlandı (%100·(eᵇ−1)).
  • Bu yorumlar ilişkiyi anlatır; nedensellik için araştırma tasarımının (deney, araç değişken vb.) buna izin vermesi gerekir.

## 3. Tez raporu

In [6]:
analiz2 = ek.panel_analizi(formul, veri, birim="firma", zaman="yil", zaman_etkisi=True)
rapor = analiz2.rapor(etiketler={"calisan": "Çalışan sayısı", "sermaye": "Sermaye stoku", "arge": "Ar-Ge yoğunluğu"})
print(rapor.paragraflar[2])

Model seçim testlerinin sonuçları Tablo 3 ile sunulmuştur. Havuzlanmış EKK ile sabit etkiler modelini karşılaştıran F testi (F(149, 1338) = 84,49; p < 0,001) birim etkilerinin anlamlı olduğunu göstermektedir. Breusch-Pagan LM testi (χ²(1) = 5.029,75; p < 0,001) rassal birim etkilerinin varlığına işaret etmektedir. Sabit ve rassal etkiler modellerini karşılaştıran Hausman testi (χ²(4) = 70,16; p < 0,001) ise birim etkilerinin açıklayıcı değişkenlerle ilişkili olduğunu, dolayısıyla rassal etkiler tahmincisinin tutarsız olduğunu göstermektedir. Test sonuçları birlikte değerlendirildiğinde sabit etkiler modelinin uygun olduğuna karar verilmiştir.


Tam rapor (Yöntem, tanımlayıcı istatistikler, model seçim testleri tablosu, üç modelin karşılaştırma tablosu, yorum, kaynakça): `analiz2.rapor("panel.docx")`
